<a href="https://colab.research.google.com/github/Apostle01/African-Cloth-Shop/blob/master/Copy_of_Workshop4_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COM6013 — Data Mining | Workshop 4
## Data Preparation: Cleaning · Integration · Reduction · Transformation

---

**How to use this notebook:**
- Read each **blue explanation box** carefully before running any code.
- Run the **Sample** cells first — they show you exactly what each function does.
- Then complete the **Your Turn** cells using the activity datasets.
- Work through every section in order — each topic builds on the last.

**Datasets needed** (upload to Colab or place in same folder):
| File | Used in |
|---|---|
| `Employee_data_Activity_1.csv` | Activity 1 — Data Cleaning |
| `Customer_Info.csv` | Activity 2 — Data Integration |
| `Customer_Sales.csv` | Activity 2 — Data Integration |
| `Workshop_4_Activity_3_Data.csv` | Activity 3 — Data Reduction |
| `Activity4_Data_Transformation.csv` | Activity 4 — Data Transformation |
| `_1_Activity.xlsx` | +1 Challenge (Advanced) |

> **Upload files to Colab:** Click the 📁 folder icon on the left → click the upload button → select your CSV files.

---

## ⚙️ Setup — Run this cell first
This installs and imports everything the notebook needs.

In [ ]:
# Run this cell first — installs and imports all required libraries
!pip install pandas numpy scikit-learn openpyxl --quiet

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")
print(f"   pandas version  : {pd.__version__}")
print(f"   numpy version   : {np.__version__}")

✅ All libraries loaded successfully!
   pandas version  : 2.2.2
   numpy version   : 2.0.2


---
# 🧹 Topic 1 — Data Cleaning

**What is data cleaning?**
Data cleaning is the process of fixing or removing:
- ❌ Missing values (empty cells, NaN)
- ❌ Duplicate rows (the same record entered twice)
- ❌ Wrong data types (a number stored as text)
- ❌ Inconsistent formatting (different date formats, mixed case)

**Why does it matter?**
If your data is dirty, your analysis will be wrong — even if your code is perfect.
Garbage in = Garbage out.

The 9 steps of data cleaning (from lesson 4 slides):
1. Identify missing data
2. Remove duplicate data
3. Correct data types
4. Standardise data formats
5. Verify data accuracy
6. Clean text data
7. Normalise and standardise data
8. Categorise data
9. Validate data consistency

---

### 📖 Sample 1A — Creating a small dirty dataset
Study this first. This shows you what messy data looks like before cleaning.

In [ ]:
# Creating a small DIRTY dataset to demonstrate cleaning functions
# This mirrors exactly what your Employee dataset looks like

dirty_data = {
    'ID':         [1,    2,    3,    4,    5,    5   ],   # row 5 duplicated!
    'Name':       ['Alice','Bob','Carol','David','Eve','Eve'],
    'Age':        [28,   None, 35,   None, 22,   22  ],   # None = missing
    'Salary':     [32000, 45000, None, 50000, None, None],
    'Department': ['Sales','HR','Sales','Finance','HR','HR']
}

df_dirty = pd.DataFrame(dirty_data)

print("=" * 45)
print("DIRTY DATASET (before cleaning):")
print("=" * 45)
print(df_dirty)
print(f"Shape: {df_dirty.shape[0]} rows × {df_dirty.shape[1]} columns")

DIRTY DATASET (before cleaning):
   ID   Name   Age   Salary Department
0   1  Alice  28.0  32000.0      Sales
1   2    Bob   NaN  45000.0         HR
2   3  Carol  35.0      NaN      Sales
3   4  David   NaN  50000.0    Finance
4   5    Eve  22.0      NaN         HR
5   5    Eve  22.0      NaN         HR
Shape: 6 rows × 5 columns


### 📖 Sample 1B — Identifying missing values with `isnull()`
Two ways to find missing values: see every cell, or count per column.

In [ ]:
# METHOD 1: See every cell — True = missing, False = present
print("Which cells are missing? (True = missing)")
print(df_dirty.isnull())
print()

# METHOD 2: Count missing values per column (more useful!)
print("How many missing values per column?")
print(df_dirty.isnull().sum())
print()

# METHOD 3: Show only rows that have at least one missing value
print("Rows with at least one missing value:")
print(df_dirty[df_dirty.isnull().any(axis=1)])

Which cells are missing? (True = missing)
      ID   Name    Age  Salary  Department
0  False  False  False   False       False
1  False  False   True   False       False
2  False  False  False    True       False
3  False  False   True   False       False
4  False  False  False    True       False
5  False  False  False    True       False

How many missing values per column?
ID            0
Name          0
Age           2
Salary        3
Department    0
dtype: int64

Rows with at least one missing value:
   ID   Name   Age   Salary Department
1   2    Bob   NaN  45000.0         HR
2   3  Carol  35.0      NaN      Sales
3   4  David   NaN  50000.0    Finance
4   5    Eve  22.0      NaN         HR
5   5    Eve  22.0      NaN         HR


### 📖 Sample 1C — Filling missing values with `fillna()`
**Rule of thumb:** Use `median` for Age (not skewed by outliers). Use `mean` for Salary.

In [ ]:
# Work on a copy so we keep the original dirty data for comparison
df_clean = df_dirty.copy()

# Before filling — check the missing counts
print("BEFORE filling:")
print(df_clean[['Age','Salary']].isnull().sum())
print()

# Fill Age with MEDIAN (middle value — not affected by extreme outliers)
'''
Here's how it's done for the df_dirty dataset's 'Age' column:

Original 'Age' values: [28, NaN, 35, NaN, 22, 22]

Non-missing 'Age' values: [28, 35, 22, 22]

Sorted non-missing 'Age' values: [22, 22, 28, 35]

Calculate median: Since there are an even number of values (4), the median is the average of the two middle values.
The middle values are 22 and 28. Therefore, the median is (22 + 28) / 2 = 25.
'''
age_median = df_clean['Age'].median()
print(f"Age median = {age_median}")
df_clean['Age'].fillna(age_median, inplace=True)

# Fill Salary with MEAN (average value)
''' The .median() method automatically ignores any NaN (missing)
values when computing the median, ensuring that only valid age entries are considered. '''
# Mean = (32000 + 45000 + 50000)/3

salary_mean = df_clean['Salary'].mean()
print(f"Salary mean = £{salary_mean:,.2f}")
df_clean['Salary'].fillna(salary_mean, inplace=True)

print()
print("AFTER filling:")
print(df_clean[['Age','Salary']].isnull().sum())
print()
print("Dataset now:")
print(df_clean)

BEFORE filling:
Age       2
Salary    3
dtype: int64

Age median = 25.0
Salary mean = £42,333.33

AFTER filling:
Age       0
Salary    0
dtype: int64

Dataset now:
   ID   Name   Age        Salary Department
0   1  Alice  28.0  32000.000000      Sales
1   2    Bob  25.0  45000.000000         HR
2   3  Carol  35.0  42333.333333      Sales
3   4  David  25.0  50000.000000    Finance
4   5    Eve  22.0  42333.333333         HR
5   5    Eve  22.0  42333.333333         HR


### 📖 Sample 1D — Removing duplicate rows with `drop_duplicates()`

In [ ]:
# Check for duplicates BEFORE removing
print(f"Rows BEFORE removing duplicates: {df_clean.shape[0]}")
print(f"Number of duplicate rows found : {df_clean.duplicated().sum()}")
print()

# Remove duplicates
df_clean = df_clean.drop_duplicates()

print(f"Rows AFTER removing duplicates : {df_clean.shape[0]}")
print()
print("Clean dataset:")
print(df_clean)
print()

# Final check — confirm no missing values remain
print("Final missing value check:")
print(df_clean.isnull().sum())

Rows BEFORE removing duplicates: 6
Number of duplicate rows found : 1

Rows AFTER removing duplicates : 5

Clean dataset:
   ID   Name   Age        Salary Department
0   1  Alice  28.0  32000.000000      Sales
1   2    Bob  25.0  45000.000000         HR
2   3  Carol  35.0  42333.333333      Sales
3   4  David  25.0  50000.000000    Finance
4   5    Eve  22.0  42333.333333         HR

Final missing value check:
ID            0
Name          0
Age           0
Salary        0
Department    0
dtype: int64


---
### ✏️ Activity 1 — Your Turn: Clean the Employee Dataset

**Dataset:** `Employee_data_Activity_1.csv`

**What you know about this dataset:**
- 1,050 rows, 5 columns: ID, Name, Age, Salary, Department
- 123 missing Age values
- 156 missing Salary values
- 50 duplicate rows

**Your tasks:**
1. Load the CSV file into a DataFrame
2. Check missing values using `isnull().sum()`
3. Fill missing **Age** values with the **median**
4. Fill missing **Salary** values with the **mean**
5. Remove duplicate rows
6. Print the shape before and after to see what changed
7. Confirm no missing values remain

> **Tip:** Follow exactly the same steps as the samples above — just replace `df_dirty` with your loaded DataFrame.


In [ ]:
# ── ACTIVITY 1: Clean the Employee Dataset ──────────────────

# STEP 1: Load the dataset
df_emp = pd.read_csv('Employee_data_Activity_1.csv')

# Preview the data


In [ ]:
# STEP 2: Check missing values


In [ ]:
# STEP 3 & 4: Fill missing values
# Fill Age with median


# Fill Salary with mean


# Verify — should now show 0 for Age and Salary


In [ ]:
# STEP 5: Remove duplicates




In [ ]:
# STEP 6 & 7: Final check and preview


---
# 🔗 Topic 2 — Data Integration

**What is data integration?**
Combining data from multiple sources into one unified table.

In the real world, your data is rarely in one place:
- Customer names → one system
- Purchase history → another system
- Loyalty card data → a third system

`pd.merge()` is how pandas joins two tables together.

**The four join types:**

| Join | Returns | Use when |
|------|---------|----------|
| `inner` | Only rows with a match in BOTH tables | You only want complete records |
| `outer` | ALL rows from both tables (NaN where no match) | You want to see all records |
| `left`  | All rows from left table, NaN if no right match | Left table is primary |
| `right` | All rows from right table, NaN if no left match | Right table is primary |

---

### 📖 Sample 2A — Creating two tables to merge
Notice that Customer 103 and 104 are in different tables — they will not appear in an inner join.

In [ ]:
# TABLE 1: Customer information
customers = pd.DataFrame({
    'Customer_ID': [101, 102, 103, 104, 105],
    'Name':        ['Alice','Bob','Carol','David','Eve'],
    'City':        ['London','Bristol','Leeds','London','Manchester']
})

# TABLE 2: Customer sales (notice 103 is missing, 106 is new)
sales = pd.DataFrame({
    'Customer_ID': [101, 102, 104, 105, 106],
    'Total_Spend': [250, 480, 120, 330, 95],
    'Last_Purchase': ['2024-01-10','2024-02-15','2024-01-22','2024-03-05','2024-02-28']
})

print("TABLE 1 — Customers:")
print(customers)
print()
print("TABLE 2 — Sales:")
print(sales)
print()
print("Notice: Customer 103 is in Customers but NOT in Sales")
print("Notice: Customer 106 is in Sales but NOT in Customers")

TABLE 1 — Customers:
   Customer_ID   Name        City
0          101  Alice      London
1          102    Bob     Bristol
2          103  Carol       Leeds
3          104  David      London
4          105    Eve  Manchester

TABLE 2 — Sales:
   Customer_ID  Total_Spend Last_Purchase
0          101          250    2024-01-10
1          102          480    2024-02-15
2          104          120    2024-01-22
3          105          330    2024-03-05
4          106           95    2024-02-28

Notice: Customer 103 is in Customers but NOT in Sales
Notice: Customer 106 is in Sales but NOT in Customers


### 📖 Sample 2B — Inner join (only matching rows)

In [ ]:
# INNER JOIN: only customers who appear in BOTH tables
merged_inner = pd.merge(customers, sales, on='Customer_ID', how='inner')

print("INNER JOIN result:")
print(merged_inner)
print(f"Rows: {merged_inner.shape[0]}")
print("→ Customer 103 (no sales) and 106 (no info) are both excluded")

INNER JOIN result:
   Customer_ID   Name        City  Total_Spend Last_Purchase
0          101  Alice      London          250    2024-01-10
1          102    Bob     Bristol          480    2024-02-15
2          104  David      London          120    2024-01-22
3          105    Eve  Manchester          330    2024-03-05
Rows: 4
→ Customer 103 (no sales) and 106 (no info) are both excluded


### 📖 Sample 2C — Outer join (all rows from both tables)

In [ ]:
# OUTER JOIN: ALL customers from BOTH tables — NaN where no match
merged_outer = pd.merge(customers, sales, on='Customer_ID', how='outer')

print("OUTER JOIN result:")
print(merged_outer)
print(f"Rows: {merged_outer.shape[0]}")
print("→ Customer 103 has NaN in sales columns (no sales data)")
print("→ Customer 106 has NaN in info columns (no customer info)")
print()
print("Missing values in merged result:")
print(merged_outer.isnull().sum())

OUTER JOIN result:
   Customer_ID   Name        City  Total_Spend Last_Purchase
0          101  Alice      London        250.0    2024-01-10
1          102    Bob     Bristol        480.0    2024-02-15
2          103  Carol       Leeds          NaN           NaN
3          104  David      London        120.0    2024-01-22
4          105    Eve  Manchester        330.0    2024-03-05
5          106    NaN         NaN         95.0    2024-02-28
Rows: 6
→ Customer 103 has NaN in sales columns (no sales data)
→ Customer 106 has NaN in info columns (no customer info)

Missing values in merged result:
Customer_ID      0
Name             1
City             1
Total_Spend      1
Last_Purchase    1
dtype: int64


### 📖 Sample 2D — Detecting mismatches between tables

In [ ]:
# Find customers in Table 1 but NOT in Table 2
only_in_customers = customers[~customers['Customer_ID'].isin(sales['Customer_ID'])]

#This creates a boolean Series (True/False) for each customer in the customers
#The ~ (tilde) operator negates this boolean Series.
#It becomes True for customers whose 'Customer_ID' is NOT in the sales DataFrame.
# only_in_customers -> selecting only those rows where the customer ID was not found in sales
print("Customers with NO sales data:")
print(only_in_customers)
print()

# Find sales records with NO matching customer info
only_in_sales = sales[~sales['Customer_ID'].isin(customers['Customer_ID'])]
print("Sales records with NO customer info:")
print(only_in_sales)

Customers with NO sales data:
   Customer_ID   Name   City
2          103  Carol  Leeds

Sales records with NO customer info:
   Customer_ID  Total_Spend Last_Purchase
4          106           95    2024-02-28


---
### ✏️ Activity 2 — Your Turn: Merge the Customer Tables

**Datasets:** `Customer_Info.csv` and `Customer_Sales.csv`

**Important discovery:** These two files have non-overlapping IDs:
- Customer_Info has IDs 1001–1100
- Customer_Sales has IDs 1051–1150
- They only SHARE IDs 1051–1100 (50 customers)

**Your tasks:**
1. Load both CSV files
2. Merge them using an **inner join** on `Customer_ID`
3. Print how many rows result — explain WHY it is not 100
4. Find which customers are in Info but not Sales
5. Find which customers are in Sales but not Info
6. Remove any duplicates from the merged result

> **Think about it:** If you used an outer join instead, how many rows would you get?


In [ ]:
# ── ACTIVITY 2: Merge Customer Tables ──────────────────────

# STEP 1: Load both datasets




In [ ]:
# STEP 2: Inner join on Customer_ID


In [ ]:
# STEP 3: Explain why you got that many rows


In [ ]:
# STEP 4 & 5: Find mismatches


In [ ]:
# STEP 6: Remove duplicates from merged result


---
# 📦 Topic 3 — Data Reduction

**What is data reduction?**
Minimising the volume of data while keeping results as close as possible to the original.

**Why do we need it?**
Machine learning models only understand **numbers**. They cannot process text like "First", "Male", or "Yes".

**Focus: Categorical Encoding**
Converting text labels into numbers so models can read them.

| Text | Encoded number |
|------|---------------|
| First | 1 |
| Second | 2 |
| Third | 3 |
| Fail | 4 |
| M (Male) | 0 |
| F (Female) | 1 |
| Y (Employed) | 1 |
| N (Not employed) | 0 |

---

### 📖 Sample 3A — Selecting categorical (text) columns with `select_dtypes()`

In [ ]:
# Create a small sample student dataset
sample_students = pd.DataFrame({
    'Student_ID': [1, 2, 3, 4, 5],
    'Gender':     ['M','F','M','F','M'],
    'Grade':      ['First','Second','Fail','First','Third'],
    'Employed':   ['Y','N','Y','Y','N'],
    'Score':      [85, 72, 41, 91, 63]   # this is already a number
})

print("Full dataset:")
print(sample_students)
print()

# select_dtypes(exclude=[np.number]) keeps ONLY text/categorical columns
# NOTE: The correct function name is select_dtypes — with an 's' at the end
df_cat = sample_students.select_dtypes(exclude=[np.number])

print("Categorical columns only (text):")
print(df_cat)
print()
print("Numeric columns only (numbers):")
print(sample_students.select_dtypes(include=[np.number]))

Full dataset:
   Student_ID Gender   Grade Employed  Score
0           1      M   First        Y     85
1           2      F  Second        N     72
2           3      M    Fail        Y     41
3           4      F   First        Y     91
4           5      M   Third        N     63

Categorical columns only (text):
  Gender   Grade Employed
0      M   First        Y
1      F  Second        N
2      M    Fail        Y
3      F   First        Y
4      M   Third        N

Numeric columns only (numbers):
   Student_ID  Score
0           1     85
1           2     72
2           3     41
3           4     91
4           5     63


### 📖 Sample 3B — Checking unique values before encoding

In [ ]:
# Always check unique values BEFORE encoding
# This tells you exactly what text values exist in each column

print("Grade unique values:   ", df_cat['Grade'].unique())
print("Gender unique values:  ", df_cat['Gender'].unique())
print("Employed unique values:", df_cat['Employed'].unique())
print()

# value_counts shows how many times each value appears
print("Grade counts:")
print(df_cat['Grade'].value_counts())

Grade unique values:    ['First' 'Second' 'Fail' 'Third']
Gender unique values:   ['M' 'F']
Employed unique values: ['Y' 'N']

Grade counts:
Grade
First     2
Second    1
Fail      1
Third     1
Name: count, dtype: int64


### 📖 Sample 3C — Encoding categories with `.replace()`

In [ ]:
# Work on a copy to preserve original
df_encoded = df_cat.copy()

# Encode Grade: First=1, Second=2, Third=3, Fail=4
df_encoded['Grade'].replace(
    {'First': 1, 'Second': 2, 'Third': 3, 'Fail': 4},
    inplace=True
)

# Encode Gender: M=0, F=1
df_encoded['Gender'].replace({'M': 0, 'F': 1}, inplace=True)

# Encode Employed: Y=1, N=0
df_encoded['Employed'].replace({'Y': 1, 'N': 0}, inplace=True)

print("BEFORE encoding:")
print(df_cat)
print()
print("AFTER encoding:")
print(df_encoded)
print()
print("Data types after encoding:")
print(df_encoded.dtypes)

BEFORE encoding:
  Gender   Grade Employed
0      M   First        Y
1      F  Second        N
2      M    Fail        Y
3      F   First        Y
4      M   Third        N

AFTER encoding:
   Gender  Grade  Employed
0       0      1         1
1       1      2         0
2       0      4         1
3       1      1         1
4       0      3         0

Data types after encoding:
Gender      int64
Grade       int64
Employed    int64
dtype: object


---
### ✏️ Activity 3 — Your Turn: Encode the Student Dataset

**Dataset:** `Workshop_4_Activity_3_Data.csv`

**What you know about this dataset:**
- 20 rows, 4 columns: Student_ID, Gender, Grade, Employed
- All columns are text/categorical (no numbers)
- Unique grades: First, Second, Third, Fail
- Unique genders: M, F
- Unique employed: Y, N

**Your tasks:**
1. Load the CSV file
2. Select only categorical columns using `select_dtypes()`
3. Check unique values in each column
4. Encode Grade → First=1, Second=2, Third=3, Fail=4
5. Encode Gender → M=0, F=1
6. Encode Employed → Y=1, N=0
7. Print the final encoded dataset

> ⚠️ **Common mistake:** The function is `select_dtypes` (with an 's') — not `select_dtype`.


In [ ]:
# ── ACTIVITY 3: Encode Categorical Variables ────────────────

# STEP 1: Load the dataset


In [ ]:
# STEP 2: Select categorical columns
# (select_dtypes — note the 's' at the end!)


In [ ]:
# STEP 3: Check unique values


In [ ]:
# STEP 4, 5, 6: Encode all three columns


---
# 🔄 Topic 4 — Data Transformation

**What is data transformation?**
Converting data from its original format into a more suitable format for analysis.

**Three techniques we use today:**

| Technique | What it does | Result |
|-----------|-------------|--------|
| **Normalisation (MinMax)** | Scales values between 0 and 1 | All values: 0.0 → 1.0 |
| **Standardisation (Z-score)** | Centres around 0, std dev = 1 | Most values: -3 → +3 |
| **Discretisation** | Groups continuous values into labelled bins | Categories: Young/Middle-aged/Senior |

**Why does this matter?**
If Age ranges 18–70 and Salary ranges 20,000–150,000,
a machine learning model thinks Salary is thousands of times more important.
Normalisation/standardisation makes them comparable.

---

### 📖 Sample 4A — Normalisation with `MinMaxScaler`
Scales values to the range 0.0 → 1.0

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Small sample dataset for demonstration
sample_transform = pd.DataFrame({
    'Name':   ['Alice','Bob','Carol','David','Eve'],
    'Age':    [22,     35,   28,    52,     41   ],
    'Salary': [25000,  48000, 35000, 85000, 62000]
})

print("Original data:")
print(sample_transform)
print()

# NORMALISATION — scales between 0 and 1
# Formula: (value - min) / (max - min)
''' MinMax scales numerical features to a specific range, typically between 0 and 1.
 This is a common technique in data preprocessing, especially for machine learning models,
 to ensure that features with larger values don't disproportionately influence the mode '''

minmax_scaler = MinMaxScaler()
sample_transform['Salary_Normalised'] = minmax_scaler.fit_transform(
    sample_transform[['Salary']]
)

print("After Normalisation (Salary → 0 to 1 scale):")
print(sample_transform[['Name','Salary','Salary_Normalised']])
print()
print(f"  Min Salary_Normalised: {sample_transform['Salary_Normalised'].min():.4f}  (lowest salary)")
print(f"  Max Salary_Normalised: {sample_transform['Salary_Normalised'].max():.4f}  (highest salary)")

Original data:
    Name  Age  Salary
0  Alice   22   25000
1    Bob   35   48000
2  Carol   28   35000
3  David   52   85000
4    Eve   41   62000

After Normalisation (Salary → 0 to 1 scale):
    Name  Salary  Salary_Normalised
0  Alice   25000           0.000000
1    Bob   48000           0.383333
2  Carol   35000           0.166667
3  David   85000           1.000000
4    Eve   62000           0.616667

  Min Salary_Normalised: 0.0000  (lowest salary)
  Max Salary_Normalised: 1.0000  (highest salary)


### 📖 Sample 4B — Standardisation with `StandardScaler`
Creates a z-score: how many standard deviations from the mean each value is.

In [ ]:
# STANDARDISATION — z-score (mean=0, std dev=1)
# Formula: (value - mean) / standard deviation

''' Standardisation technique transforms numerical features so that they have a mean of 0 and a standard deviation of 1.
It's often used when features have different scales or distributions and when the algorithm assumes
normally distributed data (like some linear models).
Standardisation takes different kinds of numbers and reshapes them so they all sit on the same scale.'''

std_scaler = StandardScaler()
sample_transform['Age_Standardised'] = std_scaler.fit_transform(
    sample_transform[['Age']]
)

print("After Standardisation (Age → z-score):")
print(sample_transform[['Name','Age','Age_Standardised']])
print()
print(f"  Mean of Age_Standardised: {sample_transform['Age_Standardised'].mean():.4f}  (should be ~0)")
print(f"  Std  of Age_Standardised: {sample_transform['Age_Standardised'].std():.4f}   (should be ~1)")
print()
print("Interpretation:")
print("  Positive z-score = above average age")
print("  Negative z-score = below average age")

After Standardisation (Age → z-score):
    Name  Age  Age_Standardised
0  Alice   22         -1.307209
1    Bob   35         -0.057671
2  Carol   28         -0.730499
3  David   52          1.576340
4    Eve   41          0.519039

  Mean of Age_Standardised: -0.0000  (should be ~0)
  Std  of Age_Standardised: 1.1180   (should be ~1)

Interpretation:
  Positive z-score = above average age
  Negative z-score = below average age


### 📖 Sample 4C — Discretisation with `pd.cut()`
Groups continuous values into meaningful labelled categories.

In [ ]:
# DISCRETISATION — group Age into named buckets
# bins define the boundaries: 0-30 = Young, 30-50 = Middle-aged, 50+ = Senior
bins   = [0,  30,          50,            100   ]
labels = ['Young', 'Middle-aged', 'Senior']

sample_transform['Age_Group'] = pd.cut(
    sample_transform['Age'],
    bins=bins,
    labels=labels
)

print("After Discretisation (Age → Age Group):")
print(sample_transform[['Name','Age','Age_Group']])
print()
print("Group counts:")
print(sample_transform['Age_Group'].value_counts().sort_index())

After Discretisation (Age → Age Group):
    Name  Age    Age_Group
0  Alice   22        Young
1    Bob   35  Middle-aged
2  Carol   28        Young
3  David   52       Senior
4    Eve   41  Middle-aged

Group counts:
Age_Group
Young          2
Middle-aged    2
Senior         1
Name: count, dtype: int64


### 📖 Sample 4D — All three transformations side by side

In [ ]:
print("COMPLETE TRANSFORMATION SUMMARY:")
print(sample_transform[['Name','Age','Age_Standardised','Age_Group',
                         'Salary','Salary_Normalised']])
print()
print("Key stats:")
print(sample_transform[['Age_Standardised','Salary_Normalised']].describe().round(4))

COMPLETE TRANSFORMATION SUMMARY:
    Name  Age  Age_Standardised    Age_Group  Salary  Salary_Normalised
0  Alice   22         -1.307209        Young   25000           0.000000
1    Bob   35         -0.057671  Middle-aged   48000           0.383333
2  Carol   28         -0.730499        Young   35000           0.166667
3  David   52          1.576340       Senior   85000           1.000000
4    Eve   41          0.519039  Middle-aged   62000           0.616667

Key stats:
       Age_Standardised  Salary_Normalised
count            5.0000             5.0000
mean            -0.0000             0.4333
std              1.1180             0.3925
min             -1.3072             0.0000
25%             -0.7305             0.1667
50%             -0.0577             0.3833
75%              0.5190             0.6167
max              1.5763             1.0000


---
### ✏️ Activity 4 — Your Turn: Transform the Employee Dataset

**Dataset:** `Activity4_Data_Transformation.csv`

**What you know about this dataset:**
- 200 rows, 5 columns: Employee_ID, Name, Age, Salary, Department
- No missing values — clean and ready for transformation
- Age range: 20–64
- Salary range: £25,412–£119,841

**Your tasks:**
1. Load the CSV file
2. Normalise **Salary** to a 0–1 scale using `MinMaxScaler`
3. Standardise **Age** to z-scores using `StandardScaler`
4. Discretise **Age** into: Young (≤30), Middle-aged (31–50), Senior (51+)
5. Print a comparison showing original and transformed values
6. Save the transformed dataset to a new CSV file


In [ ]:
# ── ACTIVITY 4: Transform the Employee Dataset ──────────────

# STEP 1: Load the dataset


In [ ]:
# STEP 2: Normalise Salary → 0 to 1


In [ ]:
# STEP 3: Standardise Age → z-score


In [ ]:
# STEP 4: Discretise Age into named groups


In [ ]:
# STEP 5: Comparison view


# STEP 6: Save to new CSV
df_transform.to_csv('Transformed_Activity4_Data.csv', index=False)

print("Transformed dataset saved as 'Transformed_Activity4_Data.csv'")

---
# 📝 Activity 5 — Reflection & Procedural Write-Up

Now that you have completed all four activities, write a detailed procedural record of what you did.

This is important because:
- It helps you consolidate what you have learned
- It will be useful when writing your assignment
- It prepares you for explaining data preparation in professional contexts

**Instructions:** In the cell below, write your answers to each question. Replace the placeholder text with your own words.

---

### ✏️ Write your answers below (double-click this cell to edit)

**1. Data Cleaning (Activity 1)**

What did you find in the Employee dataset before cleaning?
*(Example: I found X missing values in Age and Y missing values in Salary...)*

> *[Your answer here]*

Why did you use median for Age and mean for Salary?

> *[Your answer here]*

**2. Data Integration (Activity 2)**

How many rows did the inner join return, and why was it not 100?

> *[Your answer here]*

What is the difference between an inner join and an outer join?

> *[Your answer here]*

**3. Data Reduction (Activity 3)**

Why can machine learning models not use text like "First" or "Male" directly?

> *[Your answer here]*

**4. Data Transformation (Activity 4)**

What is the difference between normalisation and standardisation?

> *[Your answer here]*

When would you use discretisation instead of keeping the original numeric values?

> *[Your answer here]*

**5. Overall reflection**

If you were starting a new data science project, in what order would you apply these four preparation steps, and why?

> *[Your answer here]*


---
# 🚀 +1 Challenge — Advanced: The Messy Dataset

**Dataset:** `_1_Activity.xlsx`

This is for students who have finished all activities and want a real challenge.

This Excel file has **200 rows** and contains **6 deliberate data quality problems** hidden across 6 columns. Your job is to find and fix every one.

**The 6 problems (try to find them yourself before reading the hints!):**

| Column | Problem type |
|--------|-------------|
| CustomerID | Wrong data type |
| Age | Mixed types + missing values |
| Income | Fake NaN strings + missing values |
| City | Spelling errors + inconsistent capitalisation |
| PurchaseAmount | Fake NaN strings + missing values |
| SignupDate | Three different date formats + impossible dates |

**Your tasks:**
1. Load the Excel file
2. Explore the data to find all 6 problems
3. Fix each one
4. Verify the cleaned dataset has correct types and no issues
5. Discuss with your partner: what would happen to a machine learning model if you fed it this raw data?

> ⚠️ This dataset requires `openpyxl` — already installed in the Setup cell above.


In [ ]:
# ── +1 CHALLENGE: The Messy Dataset ────────────────────────

# Load the Excel file


In [ ]:
# EXPLORE: Check data types and missing values

# Note: Some 'NaN' values are stored as the TEXT string 'NaN'
# — isnull() will NOT catch these!


In [ ]:
# ── FIX 1: CustomerID — convert text '001' to integer ──────


In [ ]:
# ── FIX 2: Age — has text like 'thirty','forty',' twenty '
# Step 1: replace text words with NaN, then fill with median





In [ ]:
# ── FIX 3: Income — string 'NaN' not real NaN ──────────────


In [ ]:
# ── FIX 4: PurchaseAmount — string 'NaN' + missing ─────────


In [ ]:
# ── FIX 5: City — typos and inconsistent capitalisation ────


# Fix capitalisation — .str.title() makes 'london' → 'London'


# Fix known typos





In [ ]:
# ── FIX 6: SignupDate — three formats + impossible dates ───
# pd.to_datetime with errors='coerce' converts what it can
# and returns NaT (Not a Time) for impossible dates


In [ ]:
# ── FINAL CHECK ─────────────────────────────────────────────


In [ ]:
# EXTENSION: Apply transformation to the cleaned data
# Normalise Income and PurchaseAmount



---
# ✅ Workshop 4 Complete!

## What you have learned today:

| Topic | Key functions | What they do |
|-------|--------------|--------------|
| **Data Cleaning** | `isnull()`, `fillna()`, `drop_duplicates()` | Find and fix missing/duplicate data |
| **Data Integration** | `pd.merge()` | Join two tables on a common column |
| **Data Reduction** | `select_dtypes()`, `.replace()` | Encode text categories as numbers |
| **Data Transformation** | `MinMaxScaler`, `StandardScaler`, `pd.cut()` | Scale and reshape numeric data |

## Next steps:
- Review your procedural write-up in Activity 5
- Read the Week 5 iLearn material before the next session
- **Next workshop:** Logistic Regression · Decision Trees · Random Forest · Naïve Bayesian
  — everything you prepared today feeds directly into those models!

---
*COM6013 Data Mining · Workshop 4 · Data Preparation*
